In [1]:
import Levenshtein
import pandas as pd
import numpy as np
import copy

Soak up the parsed barcodes from the PacBio reads. Group them by gene, then count the number of reads per barcode.

In [2]:
barcodes = {}

df = pd.read_csv("barcodes.csv", index_col=0)
#there's a freak row that has an empty barcode entry. skipping
df = df[~df["barcode"].isna()]
for bc, tf in zip(df["barcode"], df["target"]):
    #disqualify abnormally sized BCs. we're supposed to be 12bp, let's keep it between 10 and 14
    if len(bc)<10 or len(bc)>14:
        continue
    #split ENST_NAME to get the gene name
    tf = tf.split("_")[1]
    #count up accordingly
    if tf not in barcodes:
        barcodes[tf] = {}
    if bc not in barcodes[tf]:
        barcodes[tf][bc] = 0
    barcodes[tf][bc] += 1

A general trend emerges - there seem to be some super clear forerunner barcodes and then a bunch of junk with sequencing errors.

In [3]:
barcodes["ZNF562"]

{'GTTACTGGGAGC': 232,
 'TCCAAGGCGGAT': 1013,
 'GAACTGCATGCT': 127,
 'TCGTCGCTTGCG': 271,
 'GATTGACGGATT': 76,
 'TCCAAGGCGAT': 1,
 'CTCCAAGGCGGAT': 2,
 'TCCAAGGCGGA': 2,
 'TCCAAGGCGGGAT': 1,
 'GTTACTGGGAG': 2,
 'TCCAAGGCGGATC': 1,
 'TCCAAGGGCGGAT': 2,
 'TCCAAGGCGGATCT': 1,
 'TCCAAGGGCGAT': 1,
 'GAATTGACGGATT': 1,
 'TCCAAGGCGGAGT': 1,
 'CTCGTCGCTTGCG': 1,
 'GATTTGACGGATT': 1,
 'ACCGGAACGGATG': 1}

For a start, let's turn this to a data frame (per gene), sorted in descending read count order.

In [4]:
for tf in barcodes:
    barcodes[tf] = pd.DataFrame(barcodes[tf], index=["reads"]).T.sort_values("reads", ascending=False)

Like so.

In [5]:
barcodes["ZNF562"]

,reads
TCCAAGGCGGAT,1013
TCGTCGCTTGCG,271
GTTACTGGGAGC,232
GAACTGCATGCT,127
GATTGACGGATT,76
TCCAAGGCGGA,2
TCCAAGGGCGGAT,2
GTTACTGGGAG,2
CTCCAAGGCGGAT,2
TCCAAGGCGGGAT,1


Let's define the logic of what we'll do here. Looking at the barcode list above, we can see the first blatantly obvious sequencing error, and what it originates from. Let's compute the Levenshtein distance of the two.

In [6]:
Levenshtein.distance("TCCAAGGCGGA", "TCCAAGGCGGAT")

1

Let's formulate criteria for identifying these sort of situations:
- for each gene, go through the barcodes in descending read order, comparing to everything that has more reads than it
- if a less common barcode is within Levenshtein distance 2 of a more common one, compare their exact read counts
- if the less common barcode has less than 5% of the read counts of the more common one (the performed sequencing supposedly has a 0.1% error rate per base, so across a ~12bp barcode 5% is very conservative), dub it a sequencing fluke
- delete the sequencing fluke, and any less common barcodes than it for this particular gene for safety

In [7]:
#the minimum levenshtein distance from a more common barcode for the same TF
threshold = 2
#if the ratio of the reads of the two is below this, kick out as technical effect
ratio_threshold = 0.05

breakpoints = {}
for tf in barcodes:
    if barcodes[tf].shape[0] > 1:
        #stash read count of the kicked out barcode
        breakpoints[tf] = 0
        for i in np.arange(1, barcodes[tf].shape[0]):
            #we've got a nested loop situation. if the internal loop finds the barcode to kick out
            #then this will be set and above zero and we need to break here too
            if breakpoints[tf] > 0:
                break
            for j in np.arange(0, i):
                #check the distance
                if Levenshtein.distance(barcodes[tf].index[i], barcodes[tf].index[j]) <= threshold:
                    #check the ratio
                    if barcodes[tf].iloc[i,0]/barcodes[tf].iloc[j,0] < ratio_threshold:
                        #we've got a sequencing flub. trim data frame and break proceedings
                        breakpoints[tf] = barcodes[tf].iloc[i,0]
                        barcodes[tf] = barcodes[tf].iloc[:i,:]
                        break

Let's take a look at what survives from our test gene... yep, all five primary-looking barcodes still here.

In [8]:
barcodes["ZNF562"]

,reads
TCCAAGGCGGAT,1013
TCGTCGCTTGCG,271
GTTACTGGGAGC,232
GAACTGCATGCT,127
GATTGACGGATT,76


Let's take a look at what the highest read count was for a barcode that got kicked out.

In [9]:
np.max(list(breakpoints.values()))

46

At this point we can start thinking about combining the dfs for the individual genes. Let's ensure that we've got the barcode present as an actual column, and stash the gene.

In [10]:
for tf in barcodes:
    barcodes[tf]["barcode"] = barcodes[tf].index.copy()
    barcodes[tf]["tf"] = tf

/tmp/ipykernel_2387/3198250197.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  barcodes[tf]["barcode"] = barcodes[tf].index.copy()
/tmp/ipykernel_2387/3198250197.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  barcodes[tf]["tf"] = tf


Glue it all together.

In [11]:
barcodes_combined = pd.concat(barcodes.values(), ignore_index=True)

It may not look pretty but it has all the info we need.

In [12]:
barcodes_combined

,reads,barcode,tf
0,1282,AGTTATAGTTGG,OTX2
1,717,ATCGCGATTCGG,OTX2
2,485,CTCACGGCGGTG,OTX2
3,452,TCCGTAATTCAA,OTX2
4,386,CACGCTAATCAC,OTX2
...,...,...,...
28560,2,TTGGTTGCGAAA,F5
28561,7,AGATTAATTTAC,MBD3
28562,3,CTGTTCCAGCCT,MBD3
28563,1,TTACTAGTCGTA,IGKV3D-20


Kick out any sort of duplicated barcode situations for safety. An argument could be made for trying to filter based on read count or what have you instead, but it feels better to be safe.

In [13]:
barcodes_combined = barcodes_combined[~barcodes_combined["barcode"].duplicated(keep=False)]

The vast majority survives.

In [14]:
barcodes_combined

,reads,barcode,tf
0,1282,AGTTATAGTTGG,OTX2
1,717,ATCGCGATTCGG,OTX2
2,485,CTCACGGCGGTG,OTX2
3,452,TCCGTAATTCAA,OTX2
4,386,CACGCTAATCAC,OTX2
...,...,...,...
28560,2,TTGGTTGCGAAA,F5
28561,7,AGATTAATTTAC,MBD3
28562,3,CTGTTCCAGCCT,MBD3
28563,1,TTACTAGTCGTA,IGKV3D-20


Four of our test gene barcodes survive. I went to check on the missing one and there was indeed cross-gene duplication.

```bash
ubuntu@kp9-citechonk1:/mnt/cs42/25-05-16-ivan-pacbio$ grep ",GATTGACGGATT," barcodes.csv | cut -f 3 -d , | sort |  uniq -c
    325 ENST00000350501_GRHL3
    297 ENST00000374672_KLF4
     76 ENST00000453372_ZNF562
```

In [15]:
barcodes_combined[barcodes_combined["tf"] == "ZNF562"]

,reads,barcode,tf
17,1013,TCCAAGGCGGAT,ZNF562
18,271,TCGTCGCTTGCG,ZNF562
19,232,GTTACTGGGAGC,ZNF562
20,127,GAACTGCATGCT,ZNF562


Dunk it out into a feature reference file. I double checked using SERPINA1 barcode TAGACATTAATA, a highly expressed one from last time, and it's still present and formatted correctly. So we should be good to go.

In [16]:
read = "R2"
#post-barcode sequence in revcomp, BC, pre-barcode sequence in revcomp
pattern = "CCGGAGCGAGACAAG(BC)GGTCGCTTCGGCGCG"
feature_type = "Antibody Capture"

def revcomp(seq):
    return seq.translate(str.maketrans('ATGC','TACG'))[::-1]

with open("feature_ref_guideseq.csv", "w") as fid:
    fid.writelines(["id,name,read,pattern,sequence,feature_type\n"])
    for ind in barcodes_combined.index:
        #we need to revcomp the bc
        rbc = revcomp(barcodes_combined.loc[ind, "barcode"])
        #create a naming scheme - gene-bc-counts
        name = barcodes_combined.loc[ind, "tf"]+'-'+rbc+'-'+str(barcodes_combined.loc[ind, "reads"])
        fid.writelines(",".join([name,name,read,pattern,rbc,feature_type])+"\n")